In [2]:
#### Thins the model training sample to corrrect for within-country spatial autocorrelation 
# Only applied to countries with many regions in training data
# Thinning method: repeatedly pick a random remaining region, drop every other remaining region within the threshold distance of it, and repeat 

from pathlib import Path
import pandas as pd
import os
import pandas as pd
import numpy as np

In [3]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# Import model data
DATASETS = {
    "capital": f"{cd}/Data/Clean/Training_data/capital_relative_final.csv",
    "labor":   f"{cd}/Data/Clean/Training_data/labor_relative_final.csv",
}

# Import lat/lon data
coords = pd.read_csv(f"{cd}/Data/Clean/Predictors/Vectors/lat_lon.csv")

# Set save path 
save_path = f"{cd}/Data/Clean/Training_data"

In [4]:
##### Set-up

THINNING_THRESHOLD_KM = 50   # empirically chosen from correlogram analysis
MIN_REGIONS_TO_THIN = 50      # countries with fewer regions than this are left untouched
RANDOM_SEED = 42              # change this to test sensitivity to which regions are kept

In [5]:
##### CORE FUNCTIONS

# function to calculate the haversine distance (km) between one point and an array of points
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# function to spatially thin a single country's regions
# repeatedly picks a random remaining region, drops all others within threshold_km,
# and continues until every region has either been kept or dropped
def thin_country(sub_df, threshold_km, rng):
    remaining = sub_df.copy()
    kept_rows = []

    while len(remaining) > 0:
        pick_idx = rng.choice(remaining.index.to_numpy())
        picked_row = remaining.loc[pick_idx]
        kept_rows.append(picked_row)

        dists = haversine_km(
            picked_row["lat"], picked_row["lon"],
            remaining["lat"].values, remaining["lon"].values
        )
        remaining = remaining.loc[dists >= threshold_km]

    return pd.DataFrame(kept_rows)

# function to thin an entire dataset: only thins countries with enough regions,
# leaves sparse countries untouched, and reports how much was dropped
def thin_dataset(df, threshold_km, min_regions, rng):
    thinned_parts = []
    summary_rows = []

    for country, sub in df.groupby("country_ID"):
        n_before = len(sub)

        if n_before < min_regions:
            thinned_parts.append(sub)
            summary_rows.append({
                "country_ID": country, "n_before": n_before,
                "n_after": n_before, "n_dropped": 0, "thinned": False,
            })
            continue

        thinned_sub = thin_country(sub, threshold_km, rng)
        n_after = len(thinned_sub)

        thinned_parts.append(thinned_sub)
        summary_rows.append({
            "country_ID": country, "n_before": n_before,
            "n_after": n_after, "n_dropped": n_before - n_after, "thinned": True,
        })

    thinned_df = pd.concat(thinned_parts, ignore_index=True)
    summary_df = pd.DataFrame(summary_rows).sort_values("n_dropped", ascending=False)
    return thinned_df, summary_df

In [6]:
##### Run

rng = np.random.default_rng(RANDOM_SEED)

for name, path in DATASETS.items():
    print(f"\n── {name} ──────────────────────────────")

    df = pd.read_csv(path)
    n_before_total = len(df)

    df = df.merge(coords, on="PROJ_ID", how="left")
    missing_coords = df["lat"].isna().sum()
    if missing_coords > 0:
        print(f"  warning: {missing_coords} rows missing lat/lon, dropping before thinning")
        df = df.dropna(subset=["lat", "lon"])

    thinned_df, summary_df = thin_dataset(df, THINNING_THRESHOLD_KM, MIN_REGIONS_TO_THIN, rng)

    print(summary_df.to_string(index=False))
    print(f"  total: {n_before_total:,} -> {len(thinned_df):,} rows "
          f"({n_before_total - len(thinned_df):,} dropped)")

    out_path = f"{save_path}/{name}_relative_final_thinned.csv"
    thinned_df.to_csv(out_path, index=False)
    print(f"  saved to {out_path}")


── capital ──────────────────────────────
country_ID  n_before  n_after  n_dropped  thinned
       BRA      5182     1090       4092     True
       USA      2708     1059       1649     True
       CAN      1284      272       1012     True
       ARG       413      265        148     True
       TUR        79       76          3     True
       BGR         6        6          0    False
       IND        33       33          0    False
       TZA        28       28          0    False
       SWE         3        3          0    False
       ROU         8        8          0    False
       PRT         3        3          0    False
       POL         4        4          0    False
       MEX        32       32          0    False
       ITA        21       21          0    False
       HUN         3        3          0    False
       CHN        31       31          0    False
       HRV         2        2          0    False
       BEL         2        2          0    False
       